# PaddleOCR 3.x — Google Colab

This notebook uses PaddleOCR 3.x with the new API.

**Run cells in order. Restart runtime after install.**

In [ ]:
# Step 1: Install dependencies (run once, then restart runtime)
!pip install paddlepaddle
!pip install paddleocr
!pip install langchain langchain-community

print("\n" + "="*50)
print("DONE! Now go to Runtime > Restart session")
print("Then skip this cell and run the next ones.")
print("="*50)

In [ ]:
# Step 2: Verify installation (run after restart)
import paddle
print(f"PaddlePaddle: {paddle.__version__}")

from paddleocr import PaddleOCR
print("PaddleOCR imported OK!")

In [ ]:
# Step 3: Upload your images
from google.colab import files
import os

print("Upload your card images (front_cropped.jpg, back.jpeg, etc.)")
uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} files:")
for name in uploaded.keys():
    print(f"  - {name}")

In [ ]:
# Step 4: Initialize PaddleOCR
# First run downloads models (~100MB)

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)
print("PaddleOCR initialized!")

In [ ]:
# Step 5: Test on full image
import cv2
import matplotlib.pyplot as plt

# Change this to your uploaded image name
IMAGE_PATH = "front_cropped.jpg"

# Load and display image
img = cv2.imread(IMAGE_PATH)
if img is None:
    print(f"ERROR: Could not load {IMAGE_PATH}")
    print("Available files:", os.listdir('.'))
else:
    print(f"Loaded: {IMAGE_PATH} ({img.shape[1]}x{img.shape[0]})")
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(IMAGE_PATH)
    plt.axis('off')
    plt.show()

In [ ]:
# Step 6: Run OCR on full image
result = ocr.predict(input=IMAGE_PATH)

print("OCR Results:")
print("="*60)
for res in result:
    res.print()

In [ ]:
# Step 7: Save visualization
for res in result:
    res.save_to_img("output")
    res.save_to_json("output")

print("Results saved to 'output' folder")
print("Files:", os.listdir('output') if os.path.exists('output') else 'folder not created')

---
## Field Crop Testing

Test OCR on specific field regions of the CNIE card.

In [ ]:
import numpy as np
from PIL import Image
import io

PADDING = 10

# Field definitions for 856x540 normalized card
FRONT_FIELDS = {
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45},
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48},
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45},
}

print(f"Defined {len(FRONT_FIELDS)} field regions")

In [ ]:
def test_field_paddleocr3(img, x, y, w, h, field_name, padding=PADDING):
    """
    Crop a field and run PaddleOCR 3.x on it.
    """
    ih, iw = img.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(iw, x + w + padding)
    y2 = min(ih, y + h + padding)

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        print(f"ERROR: Empty crop for {field_name}")
        return None

    # Save crop temporarily for PaddleOCR
    temp_path = f"/tmp/{field_name}_crop.jpg"
    cv2.imwrite(temp_path, crop)

    # Run OCR
    result = ocr.predict(input=temp_path)

    # Extract text
    texts = []
    if result:
        for res in result:
            if hasattr(res, 'rec_texts'):
                texts.extend(res.rec_texts)

    text = " ".join(texts) if texts else ""

    # Display
    plt.figure(figsize=(10, 1.5))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{field_name}: '{text}'")
    plt.axis('off')
    plt.show()

    return {"text": text, "field": field_name}

print("Field test function defined.")

In [ ]:
# Test all fields
print(f"Testing {len(FRONT_FIELDS)} fields")
print("="*60)

all_results = {}

for field_name, field in FRONT_FIELDS.items():
    result = test_field_paddleocr3(
        img,
        field["x"], field["y"], field["w"], field["h"],
        field_name
    )
    if result:
        all_results[field_name] = result

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for name, r in all_results.items():
    print(f"  {name:20s} -> '{r['text']}'")

---
## Alternative: Use Arabic Language Model

In [ ]:
# Initialize Arabic OCR
ocr_ar = PaddleOCR(
    lang='ar',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)
print("Arabic PaddleOCR initialized!")

In [ ]:
# Test Arabic fields
ARABIC_FIELDS = {
    "first_name_ar": {"x": 330, "y": 130, "w": 270, "h": 48},
    "last_name_ar":  {"x": 330, "y": 204, "w": 270, "h": 48},
}

print("Testing Arabic fields with Arabic model:")
print("="*60)

for field_name, field in ARABIC_FIELDS.items():
    x, y, w, h = field["x"], field["y"], field["w"], field["h"]
    
    # Crop
    crop = img[max(0,y-PADDING):min(img.shape[0],y+h+PADDING), 
               max(0,x-PADDING):min(img.shape[1],x+w+PADDING)]
    
    temp_path = f"/tmp/{field_name}_ar.jpg"
    cv2.imwrite(temp_path, crop)
    
    result = ocr_ar.predict(input=temp_path)
    
    texts = []
    if result:
        for res in result:
            if hasattr(res, 'rec_texts'):
                texts.extend(res.rec_texts)
    
    text = " ".join(texts) if texts else ""
    
    plt.figure(figsize=(8, 1.5))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{field_name}: '{text}'")
    plt.axis('off')
    plt.show()
    
    print(f"{field_name}: {text}")

---
## Back Side — MRZ Reading

In [ ]:
# Load back image (upload it first if not already)
BACK_PATH = "back.jpeg"

back = cv2.imread(BACK_PATH)
if back is None:
    print(f"Back image not found: {BACK_PATH}")
    print("Upload it using the upload cell above.")
else:
    # Rotate if needed
    if back.shape[0] > back.shape[1]:
        back = cv2.rotate(back, cv2.ROTATE_90_COUNTERCLOCKWISE)
    
    print(f"Back loaded: {back.shape[1]}x{back.shape[0]}")
    
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(back, cv2.COLOR_BGR2RGB))
    plt.title("Back Side")
    plt.axis('off')
    plt.show()

In [ ]:
# MRZ extraction
if back is not None:
    MRZ_RATIO = 0.72
    mrz_y = int(back.shape[0] * MRZ_RATIO)
    mrz_strip = back[mrz_y:, :]
    
    # Save and OCR
    cv2.imwrite("/tmp/mrz.jpg", mrz_strip)
    mrz_result = ocr.predict(input="/tmp/mrz.jpg")
    
    plt.figure(figsize=(14, 3))
    plt.imshow(cv2.cvtColor(mrz_strip, cv2.COLOR_BGR2RGB))
    plt.title("MRZ Zone")
    plt.axis('off')
    plt.show()
    
    print("\nMRZ OCR Result:")
    for res in mrz_result:
        res.print()

---
## Download Results

In [ ]:
# Download output folder as zip
import shutil

if os.path.exists('output'):
    shutil.make_archive('ocr_results', 'zip', 'output')
    files.download('ocr_results.zip')
    print("Downloaded ocr_results.zip")
else:
    print("No output folder found. Run OCR first.")